# E-commerce Delivery Intelligence

## Data Preparation

This notebook prepares the Olist e-commerce data for delivery-performance analysis and predictive modelling.

The objective is to create a clean analytical dataset where each row represents one order.

In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/olist_orders_dataset.csv")
customers = pd.read_csv("../data/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
category_translation = pd.read_csv("../data/product_category_name_translation.csv")

In [3]:
orders.shape

(99441, 8)

In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


In [5]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


In [7]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

In [8]:
delivered_orders.shape

(96478, 8)

In [9]:
delivered_orders[
    "order_delivered_customer_date"
].isna().sum()

np.int64(8)

In [10]:
delivered_orders = delivered_orders.dropna(
    subset=[
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

In [11]:
delivered_orders.shape

(96470, 8)

In [12]:
delivered_orders["is_late"] = (
    delivered_orders["order_delivered_customer_date"]
    > delivered_orders["order_estimated_delivery_date"]
).astype(int)

In [13]:
delivered_orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "is_late"
    ]
].head(10)

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,0
5,2017-07-09 21:57:05,2017-07-26 10:57:55,2017-08-01,0
7,2017-05-16 13:10:30,2017-05-26 12:55:51,2017-06-07,0
8,2017-01-23 18:29:09,2017-02-02 14:08:10,2017-03-06,0
9,2017-07-29 11:55:02,2017-08-16 17:14:30,2017-08-23,0
10,2017-05-16 19:41:10,2017-05-29 11:18:31,2017-06-07,0


In [14]:
delivered_orders["is_late"].value_counts()

is_late
0    88644
1     7826
Name: count, dtype: int64

In [15]:
delivered_orders["is_late"].mean()

np.float64(0.08112366538820359)

In [16]:
late_rate = delivered_orders["is_late"].mean() * 100

print(f"Late delivery rate: {late_rate:.2f}%")

Late delivery rate: 8.11%


In [17]:
delivered_orders["actual_delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [18]:
delivered_orders[
    ["order_purchase_timestamp",
     "order_delivered_customer_date",
     "actual_delivery_days"]
].head()

,order_purchase_timestamp,order_delivered_customer_date,actual_delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.436574
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.782037
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.394213
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.208750
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.873877


In [19]:
delivered_orders["promised_delivery_days"] = (
    delivered_orders["order_estimated_delivery_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [20]:
delivered_orders[
    ["order_purchase_timestamp",
     "order_estimated_delivery_date",
     "promised_delivery_days"]
].head()

,order_purchase_timestamp,order_estimated_delivery_date,promised_delivery_days
0,2017-10-02 10:56:33,2017-10-18,15.544063
1,2018-07-24 20:41:37,2018-08-13,19.137766
2,2018-08-08 08:38:49,2018-09-04,26.639711
3,2017-11-18 19:28:06,2017-12-15,26.188819
4,2018-02-13 21:18:39,2018-02-26,12.112049


In [21]:
delivered_orders["purchase_month"] = (
    delivered_orders["order_purchase_timestamp"].dt.month
)

In [22]:
delivered_orders[
    ["order_purchase_timestamp", "purchase_month"]
].head()

,order_purchase_timestamp,purchase_month
0,2017-10-02 10:56:33,10
1,2018-07-24 20:41:37,7
2,2018-08-08 08:38:49,8
3,2017-11-18 19:28:06,11
4,2018-02-13 21:18:39,2


In [23]:
delivered_orders["purchase_weekday"] = (
    delivered_orders["order_purchase_timestamp"].dt.day_name()
)

In [24]:
delivered_orders[
    ["order_purchase_timestamp", "purchase_weekday"]
].head()

,order_purchase_timestamp,purchase_weekday
0,2017-10-02 10:56:33,Monday
1,2018-07-24 20:41:37,Tuesday
2,2018-08-08 08:38:49,Wednesday
3,2017-11-18 19:28:06,Saturday
4,2018-02-13 21:18:39,Tuesday


In [25]:
delivered_orders["purchase_year_month"] = (
    delivered_orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

In [26]:
delivered_orders[
    ["order_purchase_timestamp", "purchase_year_month"]
].head()

,order_purchase_timestamp,purchase_year_month
0,2017-10-02 10:56:33,2017-10
1,2018-07-24 20:41:37,2018-07
2,2018-08-08 08:38:49,2018-08
3,2017-11-18 19:28:06,2017-11
4,2018-02-13 21:18:39,2018-02


In [27]:
delivered_orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'is_late',
 'actual_delivery_days',
 'promised_delivery_days',
 'purchase_month',
 'purchase_weekday',
 'purchase_year_month']

In [28]:
delivered_orders[
    [
        "actual_delivery_days",
        "promised_delivery_days"
    ]
].describe()

,actual_delivery_days,promised_delivery_days
count,96470.000000,96470.000000
mean,12.558217,23.736343
std,9.546156,8.761052
min,0.533414,2.008009
25%,6.766204,18.329905
50%,10.217477,23.230880
75%,15.720182,28.407795
max,209.628611,155.135463


### Delivery Features

For delivery-performance analysis, the dataset was restricted to completed deliveries with valid actual and estimated delivery dates.

Several features were engineered:

- `is_late`: identifies orders delivered after their estimated delivery date.
- `actual_delivery_days`: measures the time between purchase and actual delivery.
- `promised_delivery_days`: measures the delivery window originally provided to the customer.
- `purchase_month`: captures potential seasonal effects.
- `purchase_weekday`: captures potential differences by order day.
- `purchase_year_month`: supports time-series analysis.

`actual_delivery_days` will be used for descriptive analysis only and not as a predictive-model input, because the actual delivery duration would not be known when an order is placed.

In [29]:
item_summary = (
    order_items
    .groupby("order_id")
    .agg(
        order_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        number_of_items=("order_item_id", "count")
    )
    .reset_index()
)

In [30]:
item_summary.head()

,order_id,order_value,freight_value,number_of_items
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1


In [31]:
item_summary["order_id"].is_unique

True

In [32]:
master = delivered_orders.merge(
    item_summary,
    on="order_id",
    how="left"
)

In [33]:
master.shape

(96470, 17)

In [34]:
delivered_orders.shape

(96470, 14)

In [35]:
master["order_id"].is_unique

True

In [36]:
customer_info = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
]

In [37]:
master = master.merge(
    customer_info,
    on="customer_id",
    how="left"
)

In [38]:
master.shape

(96470, 20)

In [39]:
master["order_id"].is_unique

True